## Execution Example
This notebook shows the shared-factor break against `keygen_fleet`. The fleet's weak PRNG reused one prime across two devices, so their public moduli share a factor: `gcd(n_i, n_j)` reveals it instantly, and `factor_from_shared` (week-4 studio, `studios/week-04/rsa_lab.py`) turns it into the private exponent `d` for both keys.

In [1]:
from keygen_fleet import batch_gcd_recover, confirm_key, find_shared_pair
from duel1_targets import keygen_fleet

devices = keygen_fleet()
for name, key in devices.items():
    print(f"{name}: n = {key['n']}   e = {key['e']}")

device0: n = 173343701573937246797851164783237626887   e = 65537
device1: n = 238316530827066325600817667959558290561   e = 65537
device2: n = 149622071794379572412912105003214964163   e = 65537
device3: n = 167177452795823352121439935903039825453   e = 65537


The moduli look independent, but one pairwise GCD is enough to find the shared prime:

In [2]:
name_a, name_b, shared = find_shared_pair(devices)
print(f"shared prime: gcd(n_{name_a}, n_{name_b}) = {shared}")
print(f"n_{name_a} // p = {devices[name_a]['n'] // shared}")
print(f"n_{name_b} // p = {devices[name_b]['n'] // shared}")

shared prime: gcd(n_device0, n_device2) = 14723961130838400979
n_device0 // p = 11772898612920125053
n_device2 // p = 10161808392784034897


With the factorization in hand, `factor_from_shared` recovers `d` for **both** devices, and the RSA round-trip confirms each key works:

In [3]:
recovered = batch_gcd_recover(devices)
for name, d in recovered.items():
    print(f"{name}: recovered d = {d}")
    print(f"{name}: RSA round-trip ok = {confirm_key(devices[name], d)}")

device0: recovered d = 155006092543738932355592225651672081825
device0: RSA round-trip ok = True
device2: recovered d = 101591964428919489627153881052022337249
device2: RSA round-trip ok = True


**Answer**: from the public moduli alone, the shared prime `p = 14723961130838400979` falls out of `gcd(n_device0, n_device2)`, and both private keys follow: `d(device0) = 155006092543738932355592225651672081825` and `d(device2) = 101591964428919489627153881052022337249`, each confirmed by `(m^e)^d mod n == m`. Nobody factored a strong modulus: the low-entropy key generation, not the RSA math, is what broke.